In [1]:
%pip install Pillow pillow-heif

   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ------- -------------------------------- 1.0/5.5 MB 12.7 MB/s eta 0:00:01
   ---------------------------------------- 5.5/5.5 MB 30.4 MB/s  0:00:00
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ----------------------- ---------------- 4.2/7.1 MB 19.3 MB/s eta 0:00:01
   ---------------------------------------- 7.1/7.1 MB 24.2 MB/s  0:00:00

  Attempting uninstall: Pillow

    Found existing installation: pillow 10.2.0

    Uninstalling pillow-10.2.0:

      Successfully uninstalled pillow-10.2.0

   ---------------------------------------- 0/2 [Pillow]
   ---------------------------------------- 0/2 [Pillow]
   ---------------------------------------- 0/2 [Pillow]
   ---------------------------------------- 0/2 [Pillow]
   ---------------------------------------- 0/2 [Pillow]
   ---------------------------------------- 0/2 [Pillow]
   ---------------------------------------- 0/2 [Pillow]
 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyqt-tools 1.0.0 requires PyQt5, which is not installed.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\reino\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Программа для обработки фотографий из папки:
- конвертирует HEIC и PNG в JPG,
- нумерует итоговые JPG-файлы (0001.jpg, 0002.jpg, ...),
- сохраняет результат в новой папке,
- выводит статистику по разрешениям.
Исходные файлы не изменяются.
"""

import sys
import shutil
import tempfile
from pathlib import Path
from collections import defaultdict

from PIL import Image
import pillow_heif

pillow_heif.register_heif_opener()

# Расширения, которые мы умеем обрабатывать (в нижнем регистре)
SUPPORTED_EXT = {'.heic', '.png', '.jpg', '.jpeg'}


def convert_to_jpg(source_path: Path, dest_path: Path) -> bool:
    """Конвертирует или копирует изображение в JPG."""
    ext = source_path.suffix.lower()

    # Если уже JPG – копируем
    if ext in ('.jpg', '.jpeg'):
        try:
            shutil.copy2(source_path, dest_path)
            return True
        except Exception as e:
            print(f"Ошибка копирования {source_path.name}: {e}")
            return False

    # Для HEIC и PNG – конвертируем с белым фоном
    try:
        with Image.open(source_path) as img:
            if img.mode == 'RGBA':
                background = Image.new('RGB', img.size, (255, 255, 255))
                background.paste(img, mask=img.split()[3])
                img = background
            elif img.mode != 'RGB':
                img = img.convert('RGB')
            img.save(dest_path, 'JPEG', quality=95)
        return True
    except Exception as e:
        print(f"Ошибка конвертации {source_path.name}: {e}")
        return False


def get_resolution(file_path: Path) -> tuple:
    """Возвращает (ширина, высота)."""
    with Image.open(file_path) as img:
        return img.size


def print_statistics(resolutions: dict) -> None:
    """Выводит таблицу разрешений."""
    if not resolutions:
        print("Нет изображений для анализа.")
        return

    print("\n=== Статистика по разрешениям ===")
    print(f"{'Размер (ширина x высота)':<25} {'Количество':<10}")
    print("-" * 35)
    for res, count in sorted(resolutions.items(), key=lambda x: x[1], reverse=True):
        print(f"{res[0]}x{res[1]:<15} {count:<10}")
    print(f"Всего уникальных размеров: {len(resolutions)}")


def process_folder(input_folder: Path, output_folder_name: str = "numbered_jpgs") -> None:
    if not input_folder.exists() or not input_folder.is_dir():
        print(f"Ошибка: папка '{input_folder}' не существует или не является директорией.")
        sys.exit(1)

    # Выходная папка – подпапка внутри исходной
    output_folder = input_folder / output_folder_name

    print(f"Исходная папка: {input_folder.absolute()}")
    print(f"Выходная папка: {output_folder.absolute()}")

    # Перезапись выходной папки
    if output_folder.exists():
        response = input(f"Папка '{output_folder_name}' уже существует. Перезаписать? (y/n): ").strip().lower()
        if response != 'y':
            print("Операция отменена.")
            sys.exit(0)
        shutil.rmtree(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    # Собираем уникальные файлы по расширениям (без дублей по регистру)
    all_files = []
    for ext in SUPPORTED_EXT:
        # glob чувствителен к регистру на Linux, поэтому лучше искать все файлы и фильтровать
        # Но проще: используем rglob с проверкой суффикса в нижнем регистре
        for f in input_folder.iterdir():
            if f.is_file() and f.suffix.lower() in SUPPORTED_EXT:
                all_files.append(f)
    # Убираем дубликаты на случай, если какая-то файловая система вернула один файл дважды
    all_files = list(set(all_files))

    if not all_files:
        print("В исходной папке не найдено изображений (HEIC, PNG, JPG, JPEG).")
        sys.exit(0)

    print(f"Найдено уникальных файлов для обработки: {len(all_files)}")

    # Временная папка для промежуточных JPG
    with tempfile.TemporaryDirectory(dir=output_folder) as temp_dir:
        temp_path = Path(temp_dir)
        jpg_paths = []

        # Шаг 1: конвертация во временную папку
        for idx, src_path in enumerate(all_files):
            temp_jpg = temp_path / f"temp_{idx:05d}.jpg"
            if convert_to_jpg(src_path, temp_jpg):
                jpg_paths.append(temp_jpg)
                print(f"Обработан: {src_path.name} -> временный файл")
            else:
                print(f"Пропущен (ошибка): {src_path.name}")

        if not jpg_paths:
            print("Не удалось получить ни одного JPG-файла.")
            return

        # Шаг 2: сбор статистики разрешений
        resolutions = defaultdict(int)
        for jpg_file in jpg_paths:
            try:
                res = get_resolution(jpg_file)
                resolutions[res] += 1
            except Exception as e:
                print(f"Ошибка чтения разрешения {jpg_file.name}: {e}")

        print_statistics(resolutions)

        # Шаг 3: нумерация и перемещение в выходную папку
        num_digits = max(4, len(str(len(jpg_paths))))
        jpg_paths.sort(key=lambda p: p.name)

        for new_index, temp_jpg in enumerate(jpg_paths, start=1):
            new_name = f"{new_index:0{num_digits}d}.jpg"
            dest_path = output_folder / new_name
            shutil.move(str(temp_jpg), str(dest_path))
            print(f"Пронумерован: {new_name}")

        print(f"\nГотово! Пронумерованные JPG сохранены в: {output_folder}")
        print(f"Всего файлов: {len(jpg_paths)}")


def main():
    # По умолчанию папка 'footprints' (как в вашем примере)
    source_folder = Path("footprints")
    output_dir_name = "numbered_jpgs"
    process_folder(source_folder, output_dir_name)


if __name__ == "__main__":
    main()

Исходная папка: c:\Users\reino\Desktop\Учеба\Homework\Интеллектуальный анализ изображений\footprints
Выходная папка: c:\Users\reino\Desktop\Учеба\Homework\Интеллектуальный анализ изображений\footprints\numbered_jpgs
Найдено уникальных файлов для обработки: 190
Обработан: IMG_8174.HEIC -> временный файл
Обработан: 0fTV1N0yvkIY4uxMe-DY_DLVtmOyILmWdorFxbLskxbgPVODhwK0NlFzUDWzVkr8zfOsKQmXBfK_pUzpaJcl6qGC.jpg -> временный файл
Обработан: tnNW9mf1CsXpdugvZhj0kf426AHP_hWvzFSOG5xzM1nAfaZGg5fUTIiSx226qpaX66x8RJOhaUq0slrGTP629tpE.jpg -> временный файл
Обработан: IMG_8040.HEIC -> временный файл
Обработан: IMG_8140.HEIC -> временный файл
Обработан: IMG_8096.HEIC -> временный файл
Обработан: IMG_8135.HEIC -> временный файл
Обработан: IMG_8207.HEIC -> временный файл
Обработан: IMG_8097.HEIC -> временный файл
Обработан: IMG_8061.HEIC -> временный файл
Обработан: IMG_8108.HEIC -> временный файл
Обработан: IMG_8077(1).PNG -> временный файл
Обработан: IMG_8064.HEIC -> временный файл
Обработан: IMG_8082.